In [7]:
import pandas as pd
%run Caminhos.ipynb

tabela_clientes = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_clientes_parquet}')
tabela_ind_estrategico = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_ind_estrategico_parquet}')
tabela_alunos = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_alunos_parquet}')
tabela_especificacao_ind_O_E = pd.read_parquet(f'{pasta_hierarquia}{caminho_especificacao_ind_O_E_parquet}')
tabela_base_gerada = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_base_gerada_parquet}')
tabela_professor = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_professor_parquet}')
tabela_notas = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_parquet}')
tabela_notas_geral = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_notas_geral_parquet}')
tabela_frequencia_ativo = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ativo_parquet}')
tabela_frequencia_ITA = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_frequencia_ITA_parquet}')
tabela_Tempo_Integral = pd.read_parquet(f'{pasta_hierarquia}{caminho_tabela_Tempo_Integral_parquet}')

85 - % de estudantes da rede estadual com frequência regular em 75%

In [125]:
tab_qtd_freq = tabela_frequencia_ativo

## Soma as frequencias dos estudantes inscritos nas turmas ITA/IME
freq_reg = tab_qtd_freq.merge(tabela_clientes[["idMatricula", "Gerencia Regional", "Matricula_2024", "Matricula_2025", "Matricula_2026"]], on='idMatricula', how='left')
freq_reg = freq_reg.drop_duplicates(subset=["idMatricula", "Gerencia Regional"])
freq_reg_aprov = freq_reg[freq_reg['percPresenca'] >= 75]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_apr_24 = freq_reg_aprov[freq_reg_aprov['Matricula_2024'] == 1]
freq_apr_25 = freq_reg_aprov[freq_reg_aprov['Matricula_2025'] == 1]

soma_freq_apr_24 = freq_apr_24.groupby('Gerencia Regional')['idMatricula'].count().reset_index()
soma_freq_apr_25 = freq_apr_25.groupby('Gerencia Regional')['idMatricula'].count().reset_index()

# Adicionar coluna do ano
soma_freq_apr_24['Ano'] = 2024
soma_freq_apr_25['Ano'] = 2025

# Concatenar os DataFrames
soma_freq_apr = pd.concat([soma_freq_apr_24, soma_freq_apr_25])
#display(soma_freq_apr)
'----------------------------------------------------------------------------------------------------------------------------------------'
freq_reg_reprov = freq_reg[freq_reg['percPresenca'] < 75]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_rep_24 = freq_reg_reprov[freq_reg_reprov['Matricula_2024'] == 1]
freq_rep_25 = freq_reg_reprov[freq_reg_reprov['Matricula_2025'] == 1]

soma_freq_rep_24 = freq_rep_24.groupby('Gerencia Regional')['idMatricula'].count().reset_index()
soma_freq_rep_25 = freq_rep_25.groupby('Gerencia Regional')['idMatricula'].count().reset_index()

# Adicionar coluna do ano
soma_freq_rep_24['Ano'] = 2024
soma_freq_rep_25['Ano'] = 2025

# Concatenar os DataFrames
soma_freq_rep = pd.concat([soma_freq_rep_24, soma_freq_rep_25])
#display(soma_freq_rep)
'----------------------------------------------------------------------------------------------------------------------------------------'
est_freq = pd.concat([soma_freq_apr, soma_freq_rep])
#display(est_freq)
'----------------------------------------------------------------------------------------------------------------------------------------'
# Matriculas totais
sum_est_freq = est_freq.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].sum().reset_index()
sum_est_freq = sum_est_freq.rename(columns={'idMatricula': 'Qtde_Total'})
#display(sum_est_freq)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
freq_aprovativa = soma_freq_apr.merge(sum_est_freq, on=['Gerencia Regional', 'Ano'])
freq_aprovativa['% de estudantes da rede estadual com frequência regular em 75%'] = freq_aprovativa['idMatricula'] / freq_aprovativa['Qtde_Total']
freq_aprovativa = freq_aprovativa[["Gerencia Regional", "Ano", "% de estudantes da rede estadual com frequência regular em 75%"]]
display(freq_aprovativa)

,Gerencia Regional,Ano,% de estudantes da rede estadual com frequência regular em 75%
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,0.907501
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,0.927596
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,0.896689
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,0.930999
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,0.863380
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,0.879549
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,0.919713
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,0.902881
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,0.906099
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,0.914407


385 - Indicador - % frequência dos estudantes inscritos nas turmas ITA/IME

In [126]:
tab_freq = tabela_frequencia_ITA

## Qtd aulas com presença dos estudantes inscritos nas turmas ITA/IME
tab_freq = tab_freq.merge(tabela_clientes[["idMatricula", "Gerencia Regional", "Matricula_2024", "Matricula_2025", "Matricula_2026"]], on='idMatricula', how='left')

# Filtrar o DataFrame onde tipoFrequencia = Presença
tab_freq_p = tab_freq[(tab_freq['tipoFrequencia'] == "Presença")]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
freq_24 = tab_freq_p[tab_freq_p['Matricula_2024'] == 1]
freq_25 = tab_freq_p[tab_freq_p['Matricula_2025'] == 1]

soma_freq_24 = freq_24.groupby(['Gerencia Regional', 'Ano'])['idAula'].count().reset_index()
soma_freq_25 = freq_25.groupby(['Gerencia Regional', 'Ano'])['idAula'].count().reset_index()

# Concatenar os DataFrames
soma_freq = pd.concat([soma_freq_24, soma_freq_25])
soma_freq = soma_freq.rename(columns={'idAula': 'Qtde aulas com presença'})
soma_freq['Ano'] = soma_freq['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras
freq_24_total = tab_freq[tab_freq['Matricula_2024'] == 1]
freq_25_total = tab_freq[tab_freq['Matricula_2025'] == 1]

# Contar o número de idMatriculas distintas por Id Estado
estudantes_freq_24  = freq_24_total.groupby(['Gerencia Regional', 'Ano'])['idAula'].count().reset_index()
estudantes_freq_25  = freq_25_total.groupby(['Gerencia Regional', 'Ano'])['idAula'].count().reset_index()

# Concatenar os DataFrames
estudantes_freq = pd.concat([estudantes_freq_24, estudantes_freq_25])
estudantes_freq = estudantes_freq.rename(columns={'idAula': 'Qtde aulas'})
estudantes_freq['Ano'] = estudantes_freq['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_freq = soma_freq.merge(estudantes_freq, on=['Gerencia Regional', 'Ano'])
base_nota_freq['% frequência dos estudantes inscritos nas turmas ITA/IME'] = base_nota_freq['Qtde aulas com presença'] / base_nota_freq['Qtde aulas']
base_nota_freq = base_nota_freq[["Gerencia Regional", "Ano", "% frequência dos estudantes inscritos nas turmas ITA/IME"]]
display(base_nota_freq)

,Gerencia Regional,Ano,% frequência dos estudantes inscritos nas turmas ITA/IME
0,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 20ª,2024,0.999833


381 - Indicador - Média nas avaliações nas demais disciplinas

In [127]:
tab_nota_ITA = tabela_notas_geral[["Gerencia Regional", "idMatricula", "Ano", "idTurma", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular

# Filtrar o DataFrame onde turma = ITA/IME
tab_nota_ITA = tab_nota_ITA[(tab_nota_ITA['idTurma'].isin([287816, 287818]))]
# Filtrar o DataFrame onde nomeDisciplina
tab_nota_ITA = tab_nota_ITA[(~tab_nota_ITA['idDisciplina'].isin([2551, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 594, 2483, 2554, 595]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_ITA_24 = tab_nota_ITA[tab_nota_ITA['Matricula_2024'] == 1]
nota_ITA_25 = tab_nota_ITA[tab_nota_ITA['Matricula_2025'] == 1]

soma_nota_ITA_24 = nota_ITA_24.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_ITA_25 = nota_ITA_25.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_ITA = pd.concat([soma_nota_ITA_24, soma_nota_ITA_25])
soma_nota_ITA['Ano'] = soma_nota_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Id Estado
estudantes_ITA_24  = nota_ITA_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()
estudantes_ITA_25  = nota_ITA_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_ITA = pd.concat([estudantes_ITA_24, estudantes_ITA_25])
estudantes_ITA['Ano'] = estudantes_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_ITA = soma_nota_ITA.merge(estudantes_ITA, on=['Gerencia Regional', 'Ano'])
base_nota_media_ITA['Média nas avaliações nas demais disciplinas'] = base_nota_media_ITA['valorNota'] / base_nota_media_ITA['idMatricula']

base_nota_media_ITA_IME = base_nota_media_ITA.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_ITA_IME = base_nota_media_ITA_IME[(base_nota_media_ITA_IME["Id Estado"] == 3)]
display(base_nota_media_ITA_IME)

,Gerencia Regional,Ano,Média nas avaliações nas demais disciplinas
0,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 20ª,2024,7.554592


382 - Indicador - Média nas avaliações das disciplinas especificas ITA/IME

In [128]:
tab_nota_esp_ITA = tabela_notas_geral[["Gerencia Regional", "idMatricula", "idTurma", "Ano", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular
# Filtrar o DataFrame onde turma = ITA/IME
tab_nota_esp_ITA = tab_nota_esp_ITA[(tab_nota_esp_ITA['idTurma'].isin([287816, 287818]))]
# Filtrar o DataFrame onde nomeDisciplina
tab_nota_esp_ITA = tab_nota_esp_ITA[(tab_nota_esp_ITA['idDisciplina'].isin([2551, 2556, 2557, 2558, 2559, 2560, 2561, 2562, 2563, 2564, 2565, 2566, 2567, 594, 2483, 2554, 595]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_esp_ITA_24 = tab_nota_esp_ITA[tab_nota_esp_ITA['Matricula_2024'] == 1]
nota_esp_ITA_25 = tab_nota_esp_ITA[tab_nota_esp_ITA['Matricula_2025'] == 1]

soma_nota_esp_ITA_24 = nota_esp_ITA_24.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_esp_ITA_25 = nota_esp_ITA_25.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_esp_ITA = pd.concat([soma_nota_esp_ITA_24, soma_nota_esp_ITA_25])
soma_nota_esp_ITA['Ano'] = soma_nota_esp_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Id Estado
estudantes_esp_ITA_24  = nota_esp_ITA_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()
estudantes_esp_ITA_25  = nota_esp_ITA_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_esp_ITA = pd.concat([estudantes_esp_ITA_24, estudantes_esp_ITA_25])
estudantes_esp_ITA['Ano'] = estudantes_esp_ITA['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_esp_ITA = soma_nota_esp_ITA.merge(estudantes_esp_ITA, on=['Gerencia Regional', 'Ano'])
base_nota_media_esp_ITA['Média nas avaliações das disciplinas especificas ITA/IME'] = base_nota_media_esp_ITA['valorNota'] / base_nota_media_esp_ITA['idMatricula']

base_nota_media_esp_ITA_IME = base_nota_media_esp_ITA.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_esp_ITA_IME = base_nota_media_esp_ITA_IME[(base_nota_media_esp_ITA_IME["Id Estado"] == 3)]
display(base_nota_media_esp_ITA_IME)

,Gerencia Regional,Ano,Média nas avaliações das disciplinas especificas ITA/IME
0,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 20ª,2024,7.319304


402 - Indicador - Média dos alunos da rede nas disciplinas de línguas estrangeiras

In [129]:
tab_nota_LEst = tabela_notas_geral[["Gerencia Regional", "idMatricula", "Ano", "nomeDisciplina", "nomeTipoNota", "valorNota", "idDisciplina", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular Linguas Estrangeiras

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUAS ESTRANGEIRAS
nota_m_LEst = tab_nota_LEst[(tab_nota_LEst['idDisciplina'].isin([2400, 2233, 2232, 11, 10, 9]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_LEst_24 = nota_m_LEst[nota_m_LEst['Matricula_2024'] == 1]
nota_LEst_25 = nota_m_LEst[nota_m_LEst['Matricula_2025'] == 1]

soma_nota_LEst_24 = nota_LEst_24.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_LEst_25 = nota_LEst_25.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_LEst = pd.concat([soma_nota_LEst_24, soma_nota_LEst_25])
soma_nota_LEst['Ano'] = soma_nota_LEst['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente Linguas Estrangeiras

# Contar o número de idMatriculas distintas por Id Estado
estudantes_LEst_24  = nota_LEst_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()
estudantes_LEst_25  = nota_LEst_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_LEst = pd.concat([estudantes_LEst_24, estudantes_LEst_25])
soma_nota_LEst['Ano'] = soma_nota_LEst['Ano'].astype(int)
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_LEst = soma_nota_LEst.merge(estudantes_LEst, on=['Gerencia Regional', 'Ano'])
base_nota_media_LEst['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = base_nota_media_LEst['valorNota'] / base_nota_media_LEst['idMatricula']

base_nota_media_LP = base_nota_media_LEst.drop(['valorNota', 'idMatricula'], axis=1)
#base_nota_media_LP = base_nota_media_LP[(base_nota_media_LP["Id Estado"] == 3)]
display(base_nota_media_LP)

,Gerencia Regional,Ano,Média dos alunos da rede nas disciplinas de línguas estrangeiras
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,6.710637
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,6.976536
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,6.562113
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,6.950042
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,6.836349
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,6.645976
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,6.815950
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,6.721312
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,6.959216
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,7.081449


163 - Indicador - % de aprovação dos alunos do Ensino Médio - ok - A Validar

In [130]:
tab_nota_EM_Aprov_geral = tabela_notas_geral[["Gerencia Regional", "idMatricula", "Ano", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025"]]
tab_client_EM_Aprov = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_EM_Aprov_geral = tab_nota_EM_Aprov_geral.merge(tab_client_EM_Aprov, on=['idMatricula', 'Gerencia Regional', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = LÍNGUA PORTUGUESA
filt_EM = tab_nota_EM_Aprov_geral[
    (tab_nota_EM_Aprov_geral['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_EM_Aprov_geral['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_EM_Aprov_geral['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_EM_24 = filt_EM[filt_EM['Matricula_2024'] == 1]
filt_EM_25 = filt_EM[filt_EM['Matricula_2025'] == 1]

# Agrupando por "Id Escola" e "idMatricula" e calculando a média de "valorNota"
agrupar_media_24 = filt_EM_24.groupby(["Gerencia Regional", "Ano", "idMatricula", "nomeTipoNota"])["valorNota"].mean().reset_index()
agrupar_media_25 = filt_EM_25.groupby(["Gerencia Regional", "Ano", "idMatricula", "nomeTipoNota"])["valorNota"].mean().reset_index()

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [agrupar_media_25, agrupar_media_24]
mat_EM_geral = pd.concat([contar_distintos(df) for df in dataframes])

mat_EM_geral = mat_EM_geral.rename(columns={'idMatricula': 'Qtde_Matriculas'})
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_EM_aprov_geral_24 = agrupar_media_24[((agrupar_media_24['valorNota'] >= 6))]
filt_EM_aprov_geral_25 = agrupar_media_25[((agrupar_media_25['valorNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_EM_aprov_geral_25, filt_EM_aprov_geral_24]
aprovados_EM_geral = pd.concat([contar_distintos(df) for df in dataframes])

aprovados_EM_geral = aprovados_EM_geral.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Escola"
EM_geral = mat_EM_geral.merge(aprovados_EM_geral, on=['Gerencia Regional', 'Ano'])
EM_geral['Ano'] = EM_geral['Ano'].astype(int)
"-----------------------------------------------------------------------------------------------------------------"
# % de aprovação dos alunos do Ensino Médio
EM_geral['% de aprovação dos alunos do Ensino Médio'] = EM_geral['Qtde_Matriculas_Aprovadas'] / EM_geral['Qtde_Matriculas']
#EM_geral = mat_EM_geral[mat_EM_geral["Id Escola"] == 3]
EM_geral = EM_geral[['Gerencia Regional', 'Ano', '% de aprovação dos alunos do Ensino Médio']]
display(EM_geral)

,Gerencia Regional,Ano,% de aprovação dos alunos do Ensino Médio
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,0.876568
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,0.915207
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,0.850691
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,0.918592
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,0.920010
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,0.868295
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,0.903056
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,0.881543
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,0.904584
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,0.967565


1193 - Indicador - % de estudantes do E.M. com média >= 6 no componente curricular Matemática - ok - Validado

In [131]:
tab_nota_M_Aprovados = tabela_notas[["Gerencia Regional", "idMatricula", "Ano", "mediaNota", "nomeDisciplina", "Matricula_2024", "Matricula_2025"]]
tab_client_M_Aprovados = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_M_Aprovados = tab_nota_M_Aprovados.merge(tab_client_M_Aprovados, on=['idMatricula', 'Gerencia Regional', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = MATEMÁTICA
filt_M = tab_nota_M_Aprovados[
    (tab_nota_M_Aprovados['nomeDisciplina'] == "MATEMÁTICA") & 
    (tab_nota_M_Aprovados['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_M_Aprovados['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_M_Aprovados['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_M_24 = filt_M[filt_M['Matricula_2024'] == 1]
filt_M_25 = filt_M[filt_M['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_M_25, filt_M_24]
soma_nota_M = pd.concat([contar_distintos(df) for df in dataframes])
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_M_aprovados_24 = filt_M_24[((filt_M_24['mediaNota'] >= 6))]
filt_M_aprovados_25 = filt_M_25[((filt_M_25['mediaNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_M_aprovados_25, filt_M_aprovados_24]
aprovados_M = pd.concat([contar_distintos(df) for df in dataframes])

# Renomear a coluna para identificação clara
aprovados_M = aprovados_M.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Estado"
mat_M = soma_nota_M.merge(aprovados_M, on=['Gerencia Regional', 'Ano'])
mat_M['Ano'] = mat_M['Ano'].astype(int)

# Substituir valores NaN por 0 na nova coluna (caso uma escola não tenha nenhuma matrícula com nota >= 6)
mat_M['Qtde_Matriculas_Aprovadas'] = mat_M['Qtde_Matriculas_Aprovadas'].fillna(0)
"-----------------------------------------------------------------------------------------------------------------"
# % de estudantes do E.M. com média >= 6 no componente curricular Matemática
mat_M['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'] = mat_M['Qtde_Matriculas_Aprovadas'] / mat_M['idMatricula']
mat_M = mat_M[["Gerencia Regional", "Ano", "% de estudantes do E.M. com média >= 6 no componente curricular Matemática"]]
display(mat_M)

,Gerencia Regional,Ano,% de estudantes do E.M. com média >= 6 no componente curricular Matemática
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,0.810446
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,0.803364
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,0.649237
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,0.809629
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,0.749496
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,0.739846
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,0.814729
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,0.738824
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,0.743132
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,0.883028


1194 - Indicador - % de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa - ok - Validado

In [132]:
tab_nota_LP_Aprovados = tabela_notas[["Gerencia Regional", "idMatricula", "Ano", "mediaNota", "nomeDisciplina", "Matricula_2024", "Matricula_2025"]]
tab_client_LP_Aprovados = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tab_nota_LP_Aprovados = tab_nota_LP_Aprovados.merge(tab_client_LP_Aprovados, on=['idMatricula', 'Gerencia Regional', 'Ano'], how='inner')

#Filtro 'Ensino Médio' e nomeDisciplina = LÍNGUA PORTUGUESA
filt_LP = tab_nota_LP_Aprovados[
    (tab_nota_LP_Aprovados['nomeDisciplina'] == "LÍNGUA PORTUGUESA") & 
    (tab_nota_LP_Aprovados['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tab_nota_LP_Aprovados['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tab_nota_LP_Aprovados['grupo'].str.contains('Ensino', regex=False))
    ]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filt_LP_24 = filt_LP[filt_LP['Matricula_2024'] == 1]
filt_LP_25 = filt_LP[filt_LP['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_LP_25, filt_LP_24]
soma_nota_LP = pd.concat([contar_distintos(df) for df in dataframes])
"-----------------------------------------------------------------------------------------------------------------"
# Filtrar o DataFrame onde agregada = Ensino Médio
filt_LP_aprovados_24 = filt_LP_24[((filt_LP_24['mediaNota'] >= 6))]
filt_LP_aprovados_25 = filt_LP_25[((filt_LP_25['mediaNota'] >= 6))]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_LP_aprovados_25, filt_LP_aprovados_24]
aprovados_nota_LP = pd.concat([contar_distintos(df) for df in dataframes])

aprovados_nota_LP = aprovados_nota_LP.rename(columns={'idMatricula': 'Qtde_Matriculas_Aprovadas'})
"-----------------------------------------------------------------------------------------------------------------"
# Mesclar os dois DataFrames pelo "Id Estado"
nota_LP = soma_nota_LP.merge(aprovados_nota_LP, on=['Gerencia Regional', 'Ano'])
nota_LP['Ano'] = nota_LP['Ano'].astype(int)
"-----------------------------------------------------------------------------------------------------------------"
# % de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa
nota_LP['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'] = nota_LP['Qtde_Matriculas_Aprovadas'] / nota_LP['idMatricula']
#mat_LP = mat_LP[['Id Estado', '% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa']]
nota_LP = nota_LP[["Gerencia Regional", "Ano", "% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa"]]
display(nota_LP)

,Gerencia Regional,Ano,% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,0.848433
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,0.862354
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,0.775775
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,0.800784
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,0.834733
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,0.831297
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,0.826446
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,0.735577
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,0.818182
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,0.925280


2237 - Indicador - Nota média dos estudantes da Rede no componente curricular Língua Portuguesa

In [133]:
tab_nota_LP = tabela_notas_geral[["Gerencia Regional", "idMatricula", "Ano", "idDisciplina", "agregada", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular LP

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUA PORTUGUESA
nota_m_LP = tab_nota_LP[(tab_nota_LP['idDisciplina'] == 1)]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_LP_24 = nota_m_LP[nota_m_LP['Matricula_2024'] == 1]
nota_LP_25 = nota_m_LP[nota_m_LP['Matricula_2025'] == 1]

soma_nota_LP_24 = nota_LP_24.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_LP_25 = nota_LP_25.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_LP = pd.concat([soma_nota_LP_24, soma_nota_LP_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente LP

# Contar o número de idMatriculas distintas por Gerencia Regional
estudantes_LP_24  = nota_LP_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()
estudantes_LP_25  = nota_LP_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_LP = pd.concat([estudantes_LP_24, estudantes_LP_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
nota_media_LP = soma_nota_LP.merge(estudantes_LP, on=['Gerencia Regional', 'Ano'])
nota_media_LP['Ano'] = nota_media_LP['Ano'].astype(int)
nota_media_LP['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = nota_media_LP['valorNota'] / nota_media_LP['idMatricula']

nota_media_LP = nota_media_LP.drop(['valorNota', 'idMatricula'], axis=1)
#nota_media_LP = nota_media_LP[(nota_media_LP["Id Estado"] == 3)]
display(nota_media_LP)

,Gerencia Regional,Ano,Nota média dos estudantes da Rede no componente curricular Língua Portuguesa
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,6.480950
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,6.623384
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,6.016545
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,6.400269
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,6.434679
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,6.434360
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,6.492409
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,6.327788
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,6.685486
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,6.716038


2238 - Indicador - Nota média dos estudantes da Rede no componente curricular Matemática - OK

In [134]:
tab_nota_M = tabela_notas_geral[["Gerencia Regional", "idMatricula", "Ano", "idDisciplina", "agregada", "nomeTipoNota", "valorNota", "Matricula_2024", "Matricula_2025", "Matricula_2026"]]

## Soma das notas dos estudantes da Rede no componente curricular M

# Filtrar o DataFrame onde nomeDisciplina = LÍNGUA PORTUGUESA
nota_M = tab_nota_M[(tab_nota_M['idDisciplina'] == 5)]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
nota_M_24 = nota_M[nota_M['Matricula_2024'] == 1]
nota_M_25 = nota_M[nota_M['Matricula_2025'] == 1]

soma_nota_M_24 = nota_M_24.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()
soma_nota_M_25 = nota_M_25.groupby(['Gerencia Regional', 'Ano'])['valorNota'].sum().reset_index()

# Concatenar os DataFrames
soma_nota_M = pd.concat([soma_nota_M_24, soma_nota_M_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
## número de estudantes matriculados na Rede Estadual no componente M

# Contar o número de idMatriculas distintas por Id Escola
estudantes_M_24 = nota_M_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()
estudantes_M_25 = nota_M_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].count().reset_index()

# Concatenar os DataFrames
estudantes_M = pd.concat([estudantes_M_24, estudantes_M_25])
'---------------------------------------------------------------------------------------------------------------------------------------------------------------------'
# Mesclagem
base_nota_media_M = soma_nota_M.merge(estudantes_M, on=['Gerencia Regional', 'Ano'])
base_nota_media_M['Ano'] = base_nota_media_M['Ano'].astype(int)

base_nota_media_M['Nota média dos estudantes da Rede no componente curricular Matemática'] = base_nota_media_M['valorNota'] / base_nota_media_M['idMatricula']
base_nota_media_M = base_nota_media_M.drop(['valorNota', 'idMatricula'], axis=1)
display(base_nota_media_M)

,Gerencia Regional,Ano,Nota média dos estudantes da Rede no componente curricular Matemática
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,6.212869
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,6.216494
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,5.857403
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,6.353541
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,6.120672
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,6.058732
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,6.400293
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,6.127391
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,6.347022
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,6.448647


2227 - Indicador - Nº de docentes autodeclarados pretos/pardos na rede estadual

2225 - Indicador - Nº de docentes autodeclarados indígenas na rede estadual

2215 - Indicador - N° de estudantes autodeclarados pretos matriculados na rede - ok

In [135]:
tabela_alunos_est = tabela_alunos[["idAluno", "codigoEscolarAluno", "RA", "nome", "sexo", "raca"]]

# Filtrar alunos com raca 'Preta'
tabela_alunos_preta = tabela_alunos_est[tabela_alunos_est['raca'] == 'Preta']

# Mesclar os DataFrames para adicionar as colunas com o cálculo
Matricula_Alunos = tabela_alunos_preta.merge(tabela_clientes[["idAluno", "idMatricula", "Gerencia Regional", "Matricula_2026", "Matricula_2025", "Matricula_2024", "Ano"]], on=['idAluno'], how='inner')

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filter_df_24 = Matricula_Alunos[Matricula_Alunos['Matricula_2024'] == 1]
filter_df_25 = Matricula_Alunos[Matricula_Alunos['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por municipio (idMunicipio)
distinct_count_Preto_24_mun = filter_df_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_Preto_25_mun = filter_df_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames
matricula_aut_pretos = pd.concat([distinct_count_Preto_24_mun, distinct_count_Preto_25_mun])
matricula_aut_pretos.rename(columns={'idMatricula': 'N° de estudantes autodeclarados pretos matriculados na rede'}, inplace=True)
matricula_aut_pretos['Ano'] = matricula_aut_pretos['Ano'].astype(int)
display(matricula_aut_pretos)

,Gerencia Regional,Ano,N° de estudantes autodeclarados pretos matriculados na rede
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,842
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,416
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,222
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,172
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,533
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,705
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,318
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,604
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,516
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,416


2196 - Indicador - N° de estudantes autodeclarados indígenas matriculados na rede - ok - A Validar

In [136]:
# Filtrar alunos com raca 'Preta'
tabela_alunos_Ind = tabela_alunos_est[tabela_alunos_est['raca'] == 'Indígena']

num_linhas = tabela_alunos_est.shape[0]

# Mesclar os DataFrames para adicionar as colunas com o cálculo
Matricula_Alunos_Ind = tabela_alunos_Ind.merge(tabela_clientes[["idAluno", "idMatricula", "Gerencia Regional", "Matricula_2026", "Matricula_2025", "Matricula_2024", "Ano"]], on=['idAluno'], how='inner')

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filter_Ind_df_24 = Matricula_Alunos_Ind[Matricula_Alunos_Ind['Matricula_2024'] == 1]
filter_Ind_df_25 = Matricula_Alunos_Ind[Matricula_Alunos_Ind['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por escola (Id Escola)
idMatricula_Ind_24_escola = filter_Ind_df_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
idMatricula_Ind_25_escola = filter_Ind_df_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames
matricula_aut_indigena = pd.concat([idMatricula_Ind_24_escola, idMatricula_Ind_25_escola])
matricula_aut_indigena.rename(columns={'idMatricula': 'N° de estudantes autodeclarados indígenas matriculados na rede'}, inplace=True)
matricula_aut_indigena['Ano'] = matricula_aut_indigena['Ano'].astype(int)
display(matricula_aut_indigena)

,Gerencia Regional,Ano,N° de estudantes autodeclarados indígenas matriculados na rede
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,4
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,1
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,3
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,10
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,2
5,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,1
6,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,14
7,GERENCIA REGIONAL DE EDUCACAO DE PICOS,2024,1
8,GERENCIA REGIONAL DE EDUCACAO DE PIRIPIRI,2024,48
9,GERENCIA REGIONAL DE EDUCACAO DE REGENERACAO,2024,4


153 - Indicador - % de municípios com escolas da rede estadual em tempo integral

In [137]:
# Qtde de municipio com escolas da rede estadual
tabela_clientes_matricula_est = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "Id Municipio", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1
mun_24 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2024'] == 1]
mun_25 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2025'] == 1]

# Contar os valores distintos na coluna idMatricula por escola (Id Estado)
mun_24_per_mun = mun_24.groupby(['Gerencia Regional', 'Ano'])['Id Municipio'].nunique().reset_index()
mun_25_per_mun = mun_25.groupby(['Gerencia Regional', 'Ano'])['Id Municipio'].nunique().reset_index()

# Concatenar os DataFrames
combined_mun = pd.concat([mun_24_per_mun, mun_25_per_mun])

combined_mun.rename(columns={'Id Municipio': 'Qtde de municipio com escolas da rede estadual'}, inplace=True)
combined_mun['Ano'] = combined_mun['Ano'].astype(int)

In [138]:
# Quantidade de municipio com escolas da rede estadual em tempo integral
tabela_mun_integral = tabela_Tempo_Integral[["Gerencia Regional", "Id Escola", "Id Municipio", "Ano", "STATUS_INTEGRAL"]]

filt_status_mun = tabela_mun_integral[tabela_mun_integral['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_24_mun = filt_status_mun[filt_status_mun['Ano'] == 2024]
filt_25_mun = filt_status_mun[filt_status_mun['Ano'] == 2025]

# Somar os valores na coluna Status_integral por matricula (Id Municipio)
sum_24_mun = filt_24_mun.groupby(['Gerencia Regional', 'Ano'])['Id Municipio'].nunique().reset_index()
sum_25_mun = filt_25_mun.groupby(['Gerencia Regional', 'Ano'])['Id Municipio'].nunique().reset_index()

# Concatenar os Dataframes
comb_int_mun = pd.concat([sum_24_mun, sum_25_mun])
comb_int_mun.rename(columns={'Id Municipio': 'Qtde de Municipio com escolas de tempo integral'}, inplace=True)
"----------------------------------------------------------------------------------------------------------------"
# Mesclagem para DF final
resultado_mun = combined_mun.merge(comb_int_mun, on=['Gerencia Regional', 'Ano'], how='left')
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
resultado_mun['% de municípios com escolas da rede estadual em tempo integral'] = resultado_mun['Qtde de Municipio com escolas de tempo integral'] / resultado_mun['Qtde de municipio com escolas da rede estadual']
resultado_mun = resultado_mun[["Gerencia Regional", "% de municípios com escolas da rede estadual em tempo integral", "Ano"]]
display(resultado_mun)

,Gerencia Regional,% de municípios com escolas da rede estadual em tempo integral,Ano
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,0.736842,2024
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,0.285714,2024
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,0.400000,2024
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,0.230769,2024
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,0.857143,2024
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,0.692308,2024
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,0.500000,2024
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,0.272727,2024
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,0.272727,2024
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,0.700000,2024


159 - Indicador - Nº total de matrículas da rede estadual de educação

In [139]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1
filtered_df_24 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2024'] == 1]
filtered_df_23 = tabela_clientes_matricula_est[tabela_clientes_matricula_est['Matricula_2023'] == 1]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filtered_df_23, filtered_df_24]
combined_df = pd.concat([contar_distintos(df) for df in dataframes])

combined_df.rename(columns={'idMatricula': 'Nº total de matrículas da rede estadual de educação'}, inplace=True)
combined_df['Ano'] = combined_df['Ano'].astype(int)
display(combined_df)

,Gerencia Regional,Ano,Nº total de matrículas da rede estadual de educação
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2023,18344
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2023,15283
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2023,6722
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2023,12078
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2023,8210
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2023,8197
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2023,7110
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2023,5281
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2023,15198
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2023,4621


2202 - Indicador - % de matrículas de tempo integral no ensino regular, na rede estadual

In [140]:
# Quantidade de matricula de tempo integral por estado
tabela_integral = tabela_Tempo_Integral[["idMatricula", "Gerencia Regional", "Ano", "tempoIntegral"]]
tab_matr = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "Id Municipio", "Matricula_2024", "Matricula_2025"]]

# Mesclar os DataFrames mantendo apenas as colunas adicionais da tab_client_LP_Aprovados
tabela_integral = tabela_integral.merge(tab_matr, on=['idMatricula', 'Gerencia Regional', 'Ano'], how='inner')
filt_status = tabela_integral[tabela_integral['tempoIntegral'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_24 = filt_status[filt_status['Matricula_2024'] == 1]
filt_23 = filt_status[filt_status['Matricula_2025'] == 1]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_23, filt_24]
comb_int = pd.concat([contar_distintos(df) for df in dataframes])

comb_int.rename(columns={'idMatricula': 'Qtde de matricula de tempo integral'}, inplace=True)
"----------------------------------------------------------------------------------------------------------------"
# Mesclagem para DF final
resultado_final = combined_df.merge(comb_int, on=['Gerencia Regional', 'Ano'], how='inner')
resultado_final['Ano'] = resultado_final['Ano'].astype(int)
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
resultado_final['% de matrículas de tempo integral no ensino regular, na rede estadual'] = resultado_final['Qtde de matricula de tempo integral'] / resultado_final['Nº total de matrículas da rede estadual de educação']
resultado_final = resultado_final[["Gerencia Regional", "% de matrículas de tempo integral no ensino regular, na rede estadual", "Ano"]]
display(resultado_final)


,Gerencia Regional,"% de matrículas de tempo integral no ensino regular, na rede estadual",Ano
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,0.394625,2024
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,0.121523,2024
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,0.157448,2024
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,0.117925,2024
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,0.432425,2024
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,0.306204,2024
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,0.279526,2024
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,0.201465,2024
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,0.154759,2024
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,0.451268,2024


63 - Indicador - % de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)

In [141]:
# Quantidade de escolas de tempo integral por estado
tabela_integral_EPT = tabela_Tempo_Integral[["Id Escola", "Gerencia Regional", "Ano", "STATUS_INTEGRAL"]]

filt_status_TI = tabela_integral_EPT[tabela_integral_EPT['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_25_TI = filt_status_TI[filt_status_TI['Ano'] == 2025]
filt_24_TI = filt_status_TI[filt_status_TI['Ano'] == 2024]
filt_23_TI = filt_status_TI[filt_status_TI['Ano'] == 2023]
filt_22_TI = filt_status_TI[filt_status_TI['Ano'] == 2022]

base_25_TI = filt_25_TI[["Id Escola", "Gerencia Regional"]]
base_24_TI = filt_24_TI[["Id Escola", "Gerencia Regional"]]
base_23_TI = filt_23_TI[["Id Escola", "Gerencia Regional"]]
base_22_TI = filt_22_TI[["Id Escola", "Gerencia Regional"]]

# Remover duplicatas do DataFrame, mantendo apenas a primeira ocorrência
base_25_TI = base_25_TI.drop_duplicates(keep='first')
base_24_TI = base_24_TI.drop_duplicates(keep='first')
base_23_TI = base_23_TI.drop_duplicates(keep='first')
base_22_TI = base_22_TI.drop_duplicates(keep='first')

base_23_TI = pd.concat([base_23_TI, base_22_TI])
base_23_TI['Ano'] = 2023
base_24_TI = pd.concat([base_24_TI, base_23_TI])
base_24_TI['Ano'] = 2024

# Concatenar os DataFrames
df_combinado_g = pd.concat([base_24_TI, base_23_TI], ignore_index=True)
"----------------------------------------------------------------------------------------------------------------"

# Somar os valores na coluna Status_integral por matricula (Id Matricula)
base_25_TI = base_25_TI.groupby('Gerencia Regional')['Id Escola'].nunique().reset_index()
base_25_TI['Ano'] = 2025
base_24_TI = base_24_TI.groupby('Gerencia Regional')['Id Escola'].nunique().reset_index()
base_24_TI['Ano'] = 2024
base_23_TI = base_23_TI.groupby('Gerencia Regional')['Id Escola'].nunique().reset_index()
base_23_TI['Ano'] = 2023
base_22_TI = base_22_TI.groupby('Gerencia Regional')['Id Escola'].nunique().reset_index()
base_22_TI['Ano'] = 2022

# Concatenar os DataFrames
df_combinado = pd.concat([base_24_TI, base_23_TI], ignore_index=True)
df_combinado.rename(columns={'Id Escola': 'Qtde de escolas de tempo integral'}, inplace=True)

# Exibir o DataFrame combinado
display(df_combinado)

,Gerencia Regional,Qtde de escolas de tempo integral,Ano
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,31,2024
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,18,2024
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,10,2024
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,17,2024
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,19,2024
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,16,2024
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,13,2024
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,15,2024
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,21,2024
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,10,2024


In [142]:
# Tabela filtrada
tabela_clientes_TI_EPT = tabela_clientes[["Id Escola", "Gerencia Regional", "Matricula_2023", "Matricula_2024", "EtapaAgregada", "ordem"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filtered_TI_EPT_24 = tabela_clientes_TI_EPT[(tabela_clientes_TI_EPT['Matricula_2024'] == 1) & (tabela_clientes_TI_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_TI_EPT_24 = filtered_TI_EPT_24[["Id Escola", "Gerencia Regional"]]

filtered_TI_EPT_24 = filtered_TI_EPT_24.drop_duplicates(keep='first')

filtered_TI_EPT_23 = tabela_clientes_TI_EPT[(tabela_clientes_TI_EPT['Matricula_2023'] == 1) & (tabela_clientes_TI_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_TI_EPT_23 = filtered_TI_EPT_23[["Id Escola", "Gerencia Regional"]]

filtered_TI_EPT_23 = filtered_TI_EPT_23.drop_duplicates(keep='first')

# Concatenar os DataFrames
df_c_EPT = pd.concat([filtered_TI_EPT_24, filtered_TI_EPT_23], ignore_index=True)
#display(df_c_EPT)
"----------------------------------------------------------------------------------------------------------------"
# Relação inner join para DF final
TI_EPT = df_combinado_g.merge(df_c_EPT, how='inner')
TI_EPT = TI_EPT.drop_duplicates(keep='first')
#display(TI_EPT)
"----------------------------------------------------------------------------------------------------------------"
TI_EPT_24 = TI_EPT[TI_EPT['Ano'] == 2024]
#TI_EPT_24 = TI_EPT_24[["Id Estado", "Id Escola", "Ano"]]
TI_EPT_23 = TI_EPT[TI_EPT['Ano'] == 2023]

TI_EPT_24 = TI_EPT_24.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()
TI_EPT_23 = TI_EPT_23.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()

TI_TI_EPT = pd.concat([TI_EPT_24, TI_EPT_23])
TI_TI_EPT.rename(columns={'Id Escola': 'Qtde de escolas EPT em TI'}, inplace=True)

#display(TI_TI_EPT)

result_TI_EPT = TI_TI_EPT.merge(df_combinado, how='inner')
display(result_TI_EPT)
"----------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
result_TI_EPT['% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)'] = result_TI_EPT['Qtde de escolas EPT em TI'] / result_TI_EPT['Qtde de escolas de tempo integral']
result_TI_EPT = result_TI_EPT[["Gerencia Regional", "% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)", "Ano"]]
display(result_TI_EPT)
result_TI_EPT.info()

,Gerencia Regional,Ano,Qtde de escolas EPT em TI,Qtde de escolas de tempo integral
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,31,31
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,18,18
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,10,10
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,17,17
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,19,19
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,16,16
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,13,13
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,15,15
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,21,21
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,10,10


,Gerencia Regional,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),Ano
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,1.0,2024
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,1.0,2024
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,1.0,2024
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,1.0,2024
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,1.0,2024
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,1.0,2024
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,1.0,2024
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,1.0,2024
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,1.0,2024
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,1.0,2024


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 3 columns):
 #   Column                                                                                      Non-Null Count  Dtype  
---  ------                                                                                      --------------  -----  
 0   Gerencia Regional                                                                           42 non-null     object 
 1   % de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT)  42 non-null     float64
 2   Ano                                                                                         42 non-null     int64  
dtypes: float64(1), int64(1), object(1)
memory usage: 1.1+ KB


2213 - Indicador - Nº de matrículas EPT - ok - A Validar

In [143]:
# Tabela filtrada
tabela_clientes_m_ept = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "ordem"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filtered_df_EPTT_25 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2025'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_25 = filtered_df_EPTT_25[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2025"]]
filtered_df_EPTT_24 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2024'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_24 = filtered_df_EPTT_24[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2024"]]
filtered_df_EPTT_23 = tabela_clientes_m_ept[(tabela_clientes_m_ept['Matricula_2023'] == 1) & (tabela_clientes_m_ept['ordem'].isin([6, 7, 8, 13]))]
filtered_df_EPTT_23 = filtered_df_EPTT_23[["idMatricula", "Gerencia Regional", "Matricula_2023", "Ano"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "Profissional"
distinct_count_n_EPT_25 = filtered_df_EPTT_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_n_EPT_24 = filtered_df_EPTT_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
distinct_count_n_EPT_23 = filtered_df_EPTT_23.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem profissional por escola
combined_df_profissional = pd.concat([distinct_count_n_EPT_25, distinct_count_n_EPT_24, distinct_count_n_EPT_23])
combined_df_profissional.rename(columns={'idMatricula': 'Nº de matrículas EPT'}, inplace=True)
combined_df_profissional['Ano'] = combined_df_profissional['Ano'].astype(int)
display(combined_df_profissional)

,Gerencia Regional,Ano,Nº de matrículas EPT
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,7509
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,6322
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,2326
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,4674
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,3123
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,4310
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,3560
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,2621
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,6372
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,2523


72 - Indicador - Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC) - ok - A Validar

In [144]:
# Tabela filtrada
tabela_clientes_prof = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "EtapaAgregada", "NomeCurso", "NomeEtapa"]]

# Lista de textos a serem contados
cursos_filtrar = [
    "ADMINISTRAÇÃO",
    "CURSO TÉCNICO EM ARTESANATO COM ÊNFASE EM EMPREENDEDORISMO",
    "CONTROLE AMBIENTAL",
    "CURSO TÉCNICO EM DESENVOLVIMENTO DE SISTEMAS",
    "CURSO TÉCNICO EM ELETROTÉCNICA COM ÊNFASE EM ELETROMECÂNICA",
    "CURSO TÉCNICO EM ELETROTÉCNICA COM ÊNFASE EM HIDROGÊNIO VERDE",
    "CURSO TÉCNICO EM GUIA DE TURISMO",
    "CURSO TÉCNICO EM MARKETING DIGITAL",
    "CURSO TÉCNICO EM MINERAÇÃO",
    "PRODUÇÃO DE ÁUDIO E VÍDEO",
    "CURSO TÉCNICO EM PROGRAMAÇÃO DE JOGOS DIGITAIS",
    "CURSO TÉCNICO EM SISTEMAS DE ENERGIA RENOVÁVEL"
]

# Criar uma expressão regular que combine qualquer um dos textos
pattern = '|'.join(cursos_filtrar)

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
df_SEDUCTEC_25 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2025'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_25 = df_SEDUCTEC_25[["idMatricula", "Gerencia Regional", "Ano", "NomeCurso", "NomeEtapa", "Matricula_2025"]]
df_SEDUCTEC_24 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2024'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_24 = df_SEDUCTEC_24[["idMatricula", "Gerencia Regional", "Ano", "NomeCurso", "NomeEtapa", "Matricula_2024"]]
df_SEDUCTEC_23 = tabela_clientes_prof[(tabela_clientes_prof['Matricula_2023'] == 1) & (tabela_clientes_prof['NomeCurso'].str.contains(pattern))]
df_SEDUCTEC_23 = df_SEDUCTEC_23[["idMatricula", "Gerencia Regional", "Ano", "NomeCurso", "NomeEtapa", "Matricula_2023"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "Profissional"
count_SEDUCTEC_25 = df_SEDUCTEC_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
count_SEDUCTEC_24 = df_SEDUCTEC_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
count_SEDUCTEC_23 = df_SEDUCTEC_23.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem profissional por escola
combined_df_SEDUCTEC = pd.concat([count_SEDUCTEC_25, count_SEDUCTEC_24, count_SEDUCTEC_23])
combined_df_SEDUCTEC['Ano'] = combined_df_SEDUCTEC['Ano'].astype(int)
combined_df_SEDUCTEC.rename(columns={'idMatricula': 'Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'}, inplace=True)
display(combined_df_SEDUCTEC)

,Gerencia Regional,Ano,Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,3455
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,2925
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,802
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,2542
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,1460
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,2103
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,1731
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,1071
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,2985
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,1217


67 - Indicador - Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais - ok - A validar

In [145]:
# Tabela filtrada
tabela_clientes_EJA = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filtered_df_EJAT_25 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2025'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_25 = filtered_df_EJAT_25[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2025"]]
filtered_df_EJAT_24 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2024'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_24 = filtered_df_EJAT_24[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2024"]]
filtered_df_EJAT_23 = tabela_clientes_EJA[(tabela_clientes_EJA['Matricula_2023'] == 1) & (tabela_clientes_EJA['NomeEtapa'].str.contains("EJA"))]
filtered_df_EJAT_23 = filtered_df_EJAT_23[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2023"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"
EJAT_25 = filtered_df_EJAT_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
EJAT_24 = filtered_df_EJAT_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
EJAT_23 = filtered_df_EJAT_23.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem EJA por escola
combined_df_EJA = pd.concat([EJAT_25, EJAT_24, EJAT_23])
combined_df_EJA.rename(columns={'idMatricula': 'Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'}, inplace=True)
combined_df_EJA['Ano'] = combined_df_EJA['Ano'].astype(int)
display(combined_df_EJA)

,Gerencia Regional,Ano,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,4173
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,3205
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,1413
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,2618
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,1537
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,2482
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,2603
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,1302
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,2628
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,1373


2192 - Indicador - Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT) - ok - A Validar

In [146]:
# Tabela filtrada
tabela_clientes_EJA_EPT = tabela_clientes[["Gerencia Regional", "Id Escola", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filtered_m_EJA_EPT_25 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2025'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_25 = filtered_m_EJA_EPT_25[["Id Escola", "Gerencia Regional", "Ano", "Matricula_2025"]]
filtered_m_EJA_EPT_24 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2024'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_24 = filtered_m_EJA_EPT_24[["Id Escola", "Gerencia Regional", "Ano", "Matricula_2024"]]
filtered_m_EJA_EPT_23 = tabela_clientes_EJA_EPT[(tabela_clientes_EJA_EPT['Matricula_2023'] == 1) & (tabela_clientes_EJA_EPT['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_EJA_EPT['ativa'] == 1) & (tabela_clientes_EJA_EPT['ordem'].isin([6, 7, 8, 13]))]
filtered_m_EJA_EPT_23 = filtered_m_EJA_EPT_23[["Id Escola", "Gerencia Regional", "Ano", "Matricula_2023"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA" 'matriculaAtiva'
m_EJA_EPT_25 = filtered_m_EJA_EPT_25.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()
m_EJA_EPT_24 = filtered_m_EJA_EPT_24.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()
m_EJA_EPT_23 = filtered_m_EJA_EPT_23.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()

# Concatenar os DataFrames de contagem por escola
EJA_EPT = pd.concat([m_EJA_EPT_25, m_EJA_EPT_24, m_EJA_EPT_23])
EJA_EPT.rename(columns={'Id Escola': 'Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'}, inplace=True)
EJA_EPT['Ano'] = EJA_EPT['Ano'].astype(int)
display(EJA_EPT)

,Gerencia Regional,Ano,Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,28
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,20
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,14
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,18
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,16
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,19
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,14
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,14
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,23
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,12


65 - Indicador - Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)

In [147]:
# Tabela filtrada
tabela_clientes_EPT = tabela_clientes[["Gerencia Regional", "Id Escola", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_df_EPT_25 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2025'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_25 = filt_df_EPT_25[["Gerencia Regional", "Ano", "Id Escola", "Matricula_2025"]]
filt_df_EPT_24 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2024'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_24 = filt_df_EPT_24[["Gerencia Regional", "Ano", "Id Escola", "Matricula_2024"]]
filt_df_EPT_23 = tabela_clientes_EPT[(tabela_clientes_EPT['Matricula_2023'] == 1) & (tabela_clientes_EPT['ativa'] == 1) & (tabela_clientes_EPT['ordem'].isin([6, 7, 8, 13]))]
filt_df_EPT_23 = filt_df_EPT_23[["Gerencia Regional", "Ano", "Id Escola", "Matricula_2023"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_df_EPT_25 = filt_df_EPT_25.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()
filt_df_EPT_24 = filt_df_EPT_24.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()
filt_df_EPT_23 = filt_df_EPT_23.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()

EPT = pd.concat([filt_df_EPT_25, filt_df_EPT_24, filt_df_EPT_23])

# Renomear a coluna de contagem para diferenciar
EPT.rename(columns={'Id Escola': 'Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'}, inplace=True)
EPT['Ano'] = EPT['Ano'].astype(int)
display(EPT)

,Gerencia Regional,Ano,Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,45
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,34
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,22
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,27
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,26
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,25
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,20
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,23
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,34
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,16


1191 - Indicador - % de escolas da zona rural com oferta de EPT

In [148]:
# Tabela filtrada
tab_rural_EPT_tot = tabela_clientes[["Gerencia Regional", "Id Escola", "Ano", "ativa", "ordem", "localizacao", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]
tab_rural_EPT_tot = tab_rural_EPT_tot[(tab_rural_EPT_tot['ativa'] == 1) & (tab_rural_EPT_tot['localizacao'].str.contains("Rural"))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_rural_EPTT_25 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2025'] == 1)]
filt_rural_EPTT_25 = filt_rural_EPTT_25[["Gerencia Regional", "Id Escola", "Ano", "Matricula_2025"]]
filt_rural_EPTT_24 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2024'] == 1)]
filt_rural_EPTT_24 = filt_rural_EPTT_24[["Gerencia Regional", "Id Escola", "Ano", "Matricula_2024"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_rural_EPTT_25 = filt_rural_EPTT_25.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()
filt_rural_EPTT_24 = filt_rural_EPTT_24.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()

rural_EPTT = pd.concat([filt_rural_EPTT_25, filt_rural_EPTT_24])

# Remover duplicatas com base nas colunas 'Id Estado' e 'Ano'
rural_EPTT = rural_EPTT.drop_duplicates(subset=['Gerencia Regional', 'Ano'])

# Renomear a coluna de contagem para diferenciar
rural_EPTT.rename(columns={'Id Escola': 'Nº de escolas da zona rural'}, inplace=True)
rural_EPTT['Ano'] = rural_EPTT['Ano'].astype(int)
display(rural_EPTT)

,Gerencia Regional,Ano,Nº de escolas da zona rural
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,9
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,6
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,2
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,5
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,2
5,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,4
6,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,6
7,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,2
8,GERENCIA REGIONAL DE EDUCACAO DE PICOS,2024,2
9,GERENCIA REGIONAL DE EDUCACAO DE PIRIPIRI,2024,6


In [149]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "Profissional"
filt_rural_EPT_25 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2025'] == 1) & (tab_rural_EPT_tot['ordem'].isin([6, 7, 8, 13]))]
filt_rural_EPT_25 = filt_rural_EPT_25[["Gerencia Regional", "Id Escola", "Ano", "Matricula_2025"]]
filt_rural_EPT_24 = tab_rural_EPT_tot[(tab_rural_EPT_tot['Matricula_2024'] == 1) & (tab_rural_EPT_tot['ordem'].isin([6, 7, 8, 13]))]
filt_rural_EPT_24 = filt_rural_EPT_24[["Gerencia Regional", "Id Escola", "Ano", "Matricula_2024"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
filt_rural_EPT_25 = filt_rural_EPT_25.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()
filt_rural_EPT_24 = filt_rural_EPT_24.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()

rural_EPT = pd.concat([filt_rural_EPT_24, filt_rural_EPT_25])

# Remover duplicatas com base nas colunas 'Id Estado' e 'Ano'
rural_EPT = rural_EPT.drop_duplicates(subset=['Gerencia Regional', 'Ano'])

# Renomear a coluna de contagem para diferenciar
rural_EPT.rename(columns={'Id Escola': 'Nº de escolas da zona rural com oferta de EPT'}, inplace=True)
rural_EPT['Ano'] = rural_EPT['Ano'].astype(int)

rural_final = rural_EPTT.merge(rural_EPT, on=['Gerencia Regional', 'Ano'], how='left')

rural_final['% de escolas da zona rural com oferta de EPT'] = rural_final['Nº de escolas da zona rural com oferta de EPT'] / rural_final['Nº de escolas da zona rural']
rural_final = rural_final[["Gerencia Regional", "% de escolas da zona rural com oferta de EPT", "Ano"]]
display(rural_final)

,Gerencia Regional,% de escolas da zona rural com oferta de EPT,Ano
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,0.666667,2024
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,0.333333,2024
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,1.000000,2024
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,0.600000,2024
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,1.000000,2024
5,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,1.000000,2024
6,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,1.000000,2024
7,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,1.000000,2024
8,GERENCIA REGIONAL DE EDUCACAO DE PICOS,1.000000,2024
9,GERENCIA REGIONAL DE EDUCACAO DE PIRIPIRI,0.833333,2024


2275 - Indicador - % de escolas da zona rural com oferta de Tempo Integral

In [150]:
# Quantidade de escolas da zona rural com oferta de tempo integral por estado
tabela_integral_Ru = tabela_Tempo_Integral[["Id Escola", "Gerencia Regional", "localizacao", "ativa", "Ano", "STATUS_INTEGRAL"]]
tabela_integral_Ru = tabela_integral_Ru[(tabela_integral_Ru['ativa'] == 1) & (tabela_integral_Ru['localizacao'].str.contains("Rural"))]

filt_status_Ru = tabela_integral_Ru[tabela_integral_Ru['STATUS_INTEGRAL'] == 1]

# FIltrar o Dataframe onde Matricula_2024 e 2023 é igual a 1
filt_25_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2025]
filt_24_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2024]
filt_23_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2023]
filt_22_Ru = filt_status_Ru[filt_status_Ru['Ano'] == 2022]

base_25_Ru = filt_25_Ru[["Id Escola", "Gerencia Regional"]]
base_24_Ru = filt_24_Ru[["Id Escola", "Gerencia Regional"]]
base_23_Ru = filt_23_Ru[["Id Escola", "Gerencia Regional"]]
base_22_Ru = filt_22_Ru[["Id Escola", "Gerencia Regional"]]

# Remover duplicatas do DataFrame, mantendo apenas a primeira ocorrência
base_25_Ru = base_25_Ru.drop_duplicates(keep='first')
base_24_Ru = base_24_Ru.drop_duplicates(keep='first')
base_23_Ru = base_23_Ru.drop_duplicates(keep='first')
base_22_Ru = base_22_Ru.drop_duplicates(keep='first')

base_23_Ru = pd.concat([base_23_Ru, base_22_Ru])
base_23_Ru['Ano'] = 2023
base_24_Ru = pd.concat([base_24_Ru, base_23_Ru])
base_24_Ru['Ano'] = 2024

# Concatenar os DataFrames
df_combinado_Ru = pd.concat([base_24_Ru, base_23_Ru], ignore_index=True)
"----------------------------------------------------------------------------------------------------------------"

# Somar os valores na coluna Status_integral por matricula (Id Matricula)
base_25_Ru = base_25_Ru.groupby('Gerencia Regional')['Id Escola'].nunique().reset_index()
base_25_Ru['Ano'] = 2025
base_24_Ru = base_24_Ru.groupby('Gerencia Regional')['Id Escola'].nunique().reset_index()
base_24_Ru['Ano'] = 2024
base_23_Ru = base_23_Ru.groupby('Gerencia Regional')['Id Escola'].nunique().reset_index()
base_23_Ru['Ano'] = 2023

# Concatenar os DataFrames
df_combinad = pd.concat([base_24_Ru], ignore_index=True)
df_combinad.rename(columns={'Id Escola': 'Qtde de escolas da zona rural com oferta de Tempo Integral'}, inplace=True)

# Exibir o DataFrame combinado
display(df_combinad)

,Gerencia Regional,Qtde de escolas da zona rural com oferta de Tempo Integral,Ano
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,1,2024
1,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,1,2024
2,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,1,2024
3,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,1,2024
4,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2,2024
5,GERENCIA REGIONAL DE EDUCACAO DE PIRIPIRI,3,2024
6,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 19ª,1,2024
7,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 20ª,1,2024
8,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 21ª,1,2024


In [151]:
rural_final_Ru = rural_EPTT.merge(df_combinad, on=['Gerencia Regional', 'Ano'], how='left')

rural_final_Ru['% de escolas da zona rural com oferta de Tempo Integral'] = rural_final_Ru['Qtde de escolas da zona rural com oferta de Tempo Integral'] / rural_final_Ru['Nº de escolas da zona rural']
rural_final_Ru = rural_final_Ru[["Gerencia Regional", "% de escolas da zona rural com oferta de Tempo Integral", "Ano"]]
display(rural_final_Ru)

,Gerencia Regional,% de escolas da zona rural com oferta de Tempo Integral,Ano
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,0.111111,2024
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,NaN,2024
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,0.500000,2024
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,0.200000,2024
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,0.500000,2024
5,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,NaN,2024
6,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,0.333333,2024
7,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,NaN,2024
8,GERENCIA REGIONAL DE EDUCACAO DE PICOS,NaN,2024
9,GERENCIA REGIONAL DE EDUCACAO DE PIRIPIRI,0.500000,2024


2212 - Indicador - Nº de matrículas de EJA integrado ao EPT - ok - A Validar

In [152]:
# Tabela filtrada
tabela_clientes_inte = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "ativa", "ordem", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeEtapa"]]
tabela_clientes_inte = tabela_clientes_inte[(tabela_clientes_inte['NomeEtapa'].str.contains("EJA")) & (tabela_clientes_inte['ativa'] == 1) & (tabela_clientes_inte['ordem'].isin([6, 7, 8, 13]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a "EJA"
filt_df_EJA_EPT_25 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2025'] == 1)]
filt_df_EJA_EPT_25 = filt_df_EJA_EPT_25[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2025"]]
filt_df_EJA_EPT_24 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2024'] == 1)]
filt_df_EJA_EPT_24 = filt_df_EJA_EPT_24[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2024"]]
filt_df_EJA_EPT_23 = tabela_clientes_inte[(tabela_clientes_inte['Matricula_2023'] == 1)]
filt_df_EJA_EPT_23 = filt_df_EJA_EPT_23[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2023"]]

# Contar os valores distintos na coluna idMatricula por escola (idEscola) para etapa agregada igual a "EJA"  'matriculaAtiva'
dist_EJA_EPT_25 = filt_df_EJA_EPT_25.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
dist_EJA_EPT_24 = filt_df_EJA_EPT_24.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()
dist_EJA_EPT_23 = filt_df_EJA_EPT_23.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Concatenar os DataFrames de contagem por escola
combined_df_EJA_EPT = pd.concat([dist_EJA_EPT_25, dist_EJA_EPT_24, dist_EJA_EPT_23])
combined_df_EJA_EPT.rename(columns={'idMatricula': 'Nº de matrículas de EJA integrado ao EPT'}, inplace=True)
combined_df_EJA_EPT['Ano'] = combined_df_EJA_EPT['Ano'].astype(int)
display(combined_df_EJA_EPT)

,Gerencia Regional,Ano,Nº de matrículas de EJA integrado ao EPT
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,2990
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,2172
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,838
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,1289
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,1194
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,1611
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,1792
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,1060
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,2019
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,1072


2246 - Indicador - Nº de matrículas do Ensino Médio - ok - A VALIDAR

In [153]:
# Tabela filtrada
tabela_clientes_m = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL"]]

# Filtro 'Ensino Médio'
tabela_clientes_m = tabela_clientes_m[
    (tabela_clientes_m['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tabela_clientes_m['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tabela_clientes_m['grupo'].str.contains('Ensino', regex=False))
]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filt_m_25 = tabela_clientes_m[(tabela_clientes_m['Matricula_2025'] == 1)]
filt_m_25 = filt_m_25[["idMatricula", "Gerencia Regional", "Matricula_2025", "Ano", "STATUS_INTEGRAL"]]
filt_m_24 = tabela_clientes_m[(tabela_clientes_m['Matricula_2024'] == 1)]
filt_m_24 = filt_m_24[["idMatricula", "Gerencia Regional", "Matricula_2024", "Ano", "STATUS_INTEGRAL"]]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_m_25, filt_m_24]
combined_df_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_EM.rename(columns={'idMatricula': 'Nº de matrículas do Ensino Médio'}, inplace=True)
combined_df_EM['Ano'] = combined_df_EM['Ano'].astype(int)

display(combined_df_EM)

,Gerencia Regional,Ano,Nº de matrículas do Ensino Médio
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,11083
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,9744
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,3248
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,5639
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,4121
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,3984
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,2849
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,2924
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,8565
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,2378


61 - Indicador - % de matrículas do ensino médio em tempo integral

In [154]:
# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filt_m_24 = tabela_clientes_m[(tabela_clientes_m['Matricula_2024'] == 1) & (tabela_clientes_m['STATUS_INTEGRAL'] == 1)]
filt_m_24 = filt_m_24[["idMatricula", "Gerencia Regional", "Ano"]]
filt_m_25 = tabela_clientes_m[(tabela_clientes_m['Matricula_2025'] == 1) & (tabela_clientes_m['STATUS_INTEGRAL'] == 1)]
filt_m_25 = filt_m_25[["idMatricula", "Gerencia Regional", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_m_25, filt_m_24]
combined_TI_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_TI_EM.rename(columns={'idMatricula': 'Nº de matrículas do Ensino Médio em tempo integral'}, inplace=True)
"--------------------------------------------------------------------------------------------------------------------------------------------------------------------"

result_TI_EM = combined_df_EM.merge(combined_TI_EM, on=['Gerencia Regional', 'Ano'], how='left')
result_TI_EM['Ano'] = result_TI_EM['Ano'].astype(int)
"--------------------------------------------------------------------------------------------------------------------------------------------------------------------"
# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
result_TI_EM['% de matrículas do ensino médio em tempo integral'] = result_TI_EM['Nº de matrículas do Ensino Médio em tempo integral'] / result_TI_EM['Nº de matrículas do Ensino Médio']
result_TI_EM = result_TI_EM[["Gerencia Regional", "% de matrículas do ensino médio em tempo integral", "Ano"]]

display(result_TI_EM)

,Gerencia Regional,% de matrículas do ensino médio em tempo integral,Ano
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,0.350537,2024
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,0.350062,2024
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,0.350677,2024
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,0.526334,2024
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,0.488474,2024
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,0.322038,2024
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,0.509302,2024
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,0.610807,2024
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,0.423234,2024
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,0.527754,2024


2230 - Indicador - Nº de escolas com oferta de tempo integral no ensino médio

In [155]:
# N° De escolas com ensino médio
tabela_clientes_e = tabela_clientes[["Gerencia Regional", "Id Escola", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "nivelEnsino", "EtapaAgregada", "grupo", "STATUS_INTEGRAL", "tempoIntegralEscola"]]

# Filtro 'Ensino Médio'
tabela_clientes_e = tabela_clientes_e[
    (tabela_clientes_e['nivelEnsino'].str.contains('Ensino Médio', regex=False)) &
    (tabela_clientes_e['EtapaAgregada'].str.contains('Ensino Médio|Integrada', regex=True)) &
    (tabela_clientes_e['grupo'].str.contains('Ensino', regex=False)) &
    (tabela_clientes_e['tempoIntegralEscola'] == 1)
]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional e Ensino Médio
filtered_e_25 = tabela_clientes_e[(tabela_clientes_e['Matricula_2025'] == 1)]
filtered_e_25 = filtered_e_25[["Id Escola", "Gerencia Regional", "Ano"]]
filtered_e_24 = tabela_clientes_e[(tabela_clientes_e['Matricula_2024'] == 1)]
filtered_e_24 = filtered_e_24[["Id Escola", "Gerencia Regional", "Ano"]]
filtered_e_23 = tabela_clientes_e[(tabela_clientes_e['Matricula_2023'] == 1)]
filtered_e_23 = filtered_e_23[["Id Escola", "Gerencia Regional", "Ano"]]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['Id Escola'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filtered_e_25, filtered_e_24, filtered_e_23]
combined_e_EM = pd.concat([contar_distintos(df) for df in dataframes])

combined_e_EM.rename(columns={'Id Escola': 'Nº de escolas com oferta de tempo integral no ensino médio'}, inplace=True)
combined_e_EM['Ano'] = combined_e_EM['Ano'].astype(int)
display(combined_e_EM)

,Gerencia Regional,Ano,Nº de escolas com oferta de tempo integral no ensino médio
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,31
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,18
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,10
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,17
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,19
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,16
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,13
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,15
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,21
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,10


2208 - Indicador - Nº de cursos EPT ofertados - ok - A Validar

In [156]:
# Tabela filtrada
tabela_clientes_ept_of = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2023", "Matricula_2024", "Matricula_2025", "NomeCurso", "ordem"]]
tabela_clientes_ept_of = tabela_clientes_ept_of[(tabela_clientes_ept_of['ordem'].isin([6, 7, 8, 13]))]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_EPT_curso_25 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2025'] == 1)]
filt_df_EPT_curso_25 = filt_df_EPT_curso_25[["Gerencia Regional", "Ano", "Matricula_2025", "NomeCurso"]]
filt_df_EPT_curso_24 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2024'] == 1)]
filt_df_EPT_curso_24 = filt_df_EPT_curso_24[["Gerencia Regional", "Ano", "Matricula_2024", "NomeCurso"]]
filt_df_EPT_curso_23 = tabela_clientes_ept_of[(tabela_clientes_ept_of['Matricula_2023'] == 1)]
filt_df_EPT_curso_23 = filt_df_EPT_curso_23[["Gerencia Regional", "Ano", "Matricula_2023", "NomeCurso"]]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['NomeCurso'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_EPT_curso_25, filt_df_EPT_curso_24, filt_df_EPT_curso_23]
combined_df_cursos_profissional = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_cursos_profissional.rename(columns={'NomeCurso': 'Nº de cursos EPT ofertados'}, inplace=True)
combined_df_cursos_profissional['Ano'] = combined_df_cursos_profissional['Ano'].astype(int)
display(combined_df_cursos_profissional)

,Gerencia Regional,Ano,Nº de cursos EPT ofertados
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,45
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,56
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,32
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,44
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,34
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,48
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,36
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,24
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,73
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,43


2232 - Indicador - Nº de matrículas da educação especial (AEE) - ok - A Validar

In [157]:
# Tabela filtrada
tabela_clientes_AEE = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "NomeCurso", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]
tabela_clientes_AEE = tabela_clientes_AEE[(tabela_clientes_AEE['NomeCurso'] == "AEE")]

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_AEE_25 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2025'] == 1)]
filt_df_AEE_25 = filt_df_AEE_25[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2025"]]
filt_df_AEE_24 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2024'] == 1)]
filt_df_AEE_24 = filt_df_AEE_24[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2024"]]
filt_df_AEE_23 = tabela_clientes_AEE[(tabela_clientes_AEE['Matricula_2023'] == 1)]
filt_df_AEE_23 = filt_df_AEE_23[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2023"]]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_AEE_25, filt_df_AEE_24, filt_df_AEE_23]
combined_df_AEE = pd.concat([contar_distintos(df) for df in dataframes])

combined_df_AEE.rename(columns={'idMatricula': 'Nº de matrículas da educação especial (AEE)'}, inplace=True)
combined_df_AEE['Ano'] = combined_df_AEE['Ano'].astype(int)
display(combined_df_AEE)

,Gerencia Regional,Ano,Nº de matrículas da educação especial (AEE)
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2024,75
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2024,108
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2024,66
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2024,48
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2024,24
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2024,44
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2024,57
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2024,41
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2024,143
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2024,39


348 - Indicadores - Nº de matrículas transferidas da rede estadual para a rede municipal - ok - A Validar

In [158]:
# Tabela filtrada
tabela_clientes_E_M = tabela_clientes[["idMatricula", "Gerencia Regional", "Ano", "NomeEtapa", "Matricula_2023", "Matricula_2024", "Matricula_2025"]]

Etapa_filtrar = [
    "Ensino fundamental de 9 anos - 5º Ano",
    "Ensino fundamental de 9 anos - 6º Ano",
    "Ensino fundamental de 9 anos - 7º Ano",
    "Ensino fundamental de 9 anos - 8º Ano",
    "Ensino fundamental de 9 anos - 9º Ano"
]

# Criar uma expressão regular que combine qualquer um dos textos
pattern = '|'.join(Etapa_filtrar)

# Filtrar o DataFrame onde Matricula_2024 é igual a 1 e etapa agregada igual a profissional
filt_df_E_M_25 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2025'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_25 = filt_df_E_M_25[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2025"]]
filt_df_E_M_24 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2024'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_24 = filt_df_E_M_24[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2024"]]
filt_df_E_M_23 = tabela_clientes_E_M[(tabela_clientes_E_M['Matricula_2023'] == 1) & (tabela_clientes_E_M['NomeEtapa'].str.contains(pattern))]
filt_df_E_M_23 = filt_df_E_M_23[["idMatricula", "Gerencia Regional", "Ano", "Matricula_2023"]]

def contar_distintos(df):
    return df.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].nunique().reset_index()

# Aplicar a função e concatenar os DataFrames de uma vez
dataframes = [filt_df_E_M_25, filt_df_E_M_24, filt_df_E_M_23]
combined_df_E_M = pd.concat([contar_distintos(df) for df in dataframes])

# Exibir o DataFrame combinado
combined_df_E_M = combined_df_E_M.groupby(['Gerencia Regional', 'Ano'])['idMatricula'].sum().reset_index()
combined_df_E_M['Ano'] = combined_df_E_M['Ano'].astype(int)

# Ordena o DataFrame pelo ano (caso não esteja ordenado)
combined_df_E_M = combined_df_E_M.sort_values(by='Ano')

# Calcula a diferença de matrículas ano a ano
combined_df_E_M['Diferença de matrículas'] = combined_df_E_M['idMatricula'].diff()
combined_df_E_M.rename(columns={'Diferença de matrículas': 'Nº de matrículas transferidas da rede estadual para a rede municipal'}, inplace=True)
combined_df_E_M = combined_df_E_M[['Gerencia Regional', 'Ano', 'Nº de matrículas transferidas da rede estadual para a rede municipal']]
display(combined_df_E_M)

,Gerencia Regional,Ano,Nº de matrículas transferidas da rede estadual para a rede municipal
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2023,NaN
22,GERENCIA REGIONAL DE EDUCACAO DE REGENERACAO,2023,-418.0
28,GERENCIA REGIONAL DE EDUCACAO DE TERESINA,2023,1860.0
20,GERENCIA REGIONAL DE EDUCACAO DE PIRIPIRI,2023,-1375.0
38,GERENCIA REGIONAL DE EDUCACAO DE VALENCA,2023,-192.0
18,GERENCIA REGIONAL DE EDUCACAO DE PICOS,2023,839.0
16,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2023,1823.0
30,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 19ª,2023,-977.0
14,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2023,-2437.0
24,GERENCIA REGIONAL DE EDUCACAO DE SAO JOAO DO P...,2023,53.0


Mesclagem de DateFrame Parcial

In [159]:
# Mesclar os DataFrames para adicionar as colunas com o 
# 159 - 2213
final_combined_df = combined_df.merge(combined_df_profissional, on=['Gerencia Regional', 'Ano'], how='left')
#67
final_combined_df = final_combined_df.merge(combined_df_EJA, on=['Gerencia Regional', 'Ano'], how='left')
#2246
final_combined_df = final_combined_df.merge(combined_df_EM, on=['Gerencia Regional', 'Ano'], how='left')
#2208
final_combined_df = final_combined_df.merge(combined_df_cursos_profissional, on=['Gerencia Regional', 'Ano'], how='left')
#2232
final_combined_df = final_combined_df.merge(combined_df_AEE, on=['Gerencia Regional', 'Ano'], how='left')
#72
final_combined_df = final_combined_df.merge(combined_df_SEDUCTEC, on=['Gerencia Regional', 'Ano'], how='left')
#2192
final_combined_df = final_combined_df.merge(EJA_EPT, on=['Gerencia Regional', 'Ano'], how='left')
#348
final_combined_df = final_combined_df.merge(combined_df_E_M, on=['Gerencia Regional', 'Ano'], how='left')
#65
final_combined_df = final_combined_df.merge(EPT, on=['Gerencia Regional', 'Ano'], how='left')
#2196
final_combined_df = final_combined_df.merge(matricula_aut_indigena, on=['Gerencia Regional', 'Ano'], how='left')
#2215
final_combined_df = final_combined_df.merge(matricula_aut_pretos, on=['Gerencia Regional', 'Ano'], how='left')
#2238
final_combined_df = final_combined_df.merge(base_nota_media_M, on=['Gerencia Regional', 'Ano'], how='left')
#402
final_combined_df = final_combined_df.merge(base_nota_media_LP, on=['Gerencia Regional', 'Ano'], how='left')
#1194
final_combined_df = final_combined_df.merge(nota_LP, on=['Gerencia Regional', 'Ano'], how='left')
#1193
final_combined_df = final_combined_df.merge(mat_M, on=['Gerencia Regional', 'Ano'], how='left')
#163
final_combined_df = final_combined_df.merge(EM_geral, on=['Gerencia Regional', 'Ano'], how='left')
#2237
final_combined_df = final_combined_df.merge(nota_media_LP, on=['Gerencia Regional', 'Ano'], how='left')
#381
final_combined_df = final_combined_df.merge(base_nota_media_ITA_IME, on=['Gerencia Regional', 'Ano'], how='left')
#382
final_combined_df = final_combined_df.merge(base_nota_media_esp_ITA_IME, on=['Gerencia Regional', 'Ano'], how='left')
#385
final_combined_df = final_combined_df.merge(base_nota_freq, on=['Gerencia Regional', 'Ano'], how='left')
#2202
final_combined_df = final_combined_df.merge(resultado_final, on=['Gerencia Regional', 'Ano'], how='left')
#2212
final_combined_df = final_combined_df.merge(combined_df_EJA_EPT, on=['Gerencia Regional', 'Ano'], how='left')
#2230
final_combined_df = final_combined_df.merge(combined_e_EM, on=['Gerencia Regional', 'Ano'], how='left')
#63
final_combined_df = final_combined_df.merge(result_TI_EPT, on=['Gerencia Regional', 'Ano'], how='left')
#153
final_combined_df = final_combined_df.merge(resultado_mun, on=['Gerencia Regional', 'Ano'], how='left')
#61
final_combined_df = final_combined_df.merge(result_TI_EM, on=['Gerencia Regional', 'Ano'], how='left')
#85
final_combined_df = final_combined_df.merge(freq_aprovativa, on=['Gerencia Regional', 'Ano'], how='left')
#1191
final_combined_df = final_combined_df.merge(rural_final, on=['Gerencia Regional', 'Ano'], how='left')
#2275
final_combined_df = final_combined_df.merge(rural_final_Ru, on=['Gerencia Regional', 'Ano'], how='left')
"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"

# Substituindo os NaN por 0 (ou outro valor apropriado)
#2230
final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'] = final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'].fillna(0)
#2212
final_combined_df['Nº de matrículas de EJA integrado ao EPT'] = final_combined_df['Nº de matrículas de EJA integrado ao EPT'].fillna(0)
#2202
final_combined_df['% de matrículas de tempo integral no ensino regular, na rede estadual'] = final_combined_df['% de matrículas de tempo integral no ensino regular, na rede estadual'].fillna(0)
#1194
final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'] = final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Língua Portuguesa'].fillna(0)
#1193
final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'] = final_combined_df['% de estudantes do E.M. com média >= 6 no componente curricular Matemática'].fillna(0)
#381
final_combined_df['Média nas avaliações nas demais disciplinas'] = final_combined_df['Média nas avaliações nas demais disciplinas'].fillna(0)
#163
final_combined_df['% de aprovação dos alunos do Ensino Médio'] = final_combined_df['% de aprovação dos alunos do Ensino Médio'].fillna(0)
#385
final_combined_df['% frequência dos estudantes inscritos nas turmas ITA/IME'] = final_combined_df['% frequência dos estudantes inscritos nas turmas ITA/IME'].fillna(0)
#382
final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'] = final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'].fillna(0)
#402
final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'].fillna(0)
#2237
final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'].fillna(0)
#2238
final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'].fillna(0)
#2215
final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'].fillna(0)
#2196
final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'].fillna(0)
#65
final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'].fillna(0)
#348
final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'] = final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'].fillna(0)
#2192
final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'].fillna(0)
#72
final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'] = final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'].fillna(0)
#2232
final_combined_df['Nº de matrículas da educação especial (AEE)'] = final_combined_df['Nº de matrículas da educação especial (AEE)'].fillna(0)
#2213
final_combined_df['Nº de matrículas EPT'] = final_combined_df['Nº de matrículas EPT'].fillna(0)
#67
final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'] = final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'].fillna(0)
#2246
final_combined_df['Nº de matrículas do Ensino Médio'] = final_combined_df['Nº de matrículas do Ensino Médio'].fillna(0)
#2208
final_combined_df['Nº de cursos EPT ofertados'] = final_combined_df['Nº de cursos EPT ofertados'].fillna(0)
#153
final_combined_df['% de municípios com escolas da rede estadual em tempo integral'] = final_combined_df['% de municípios com escolas da rede estadual em tempo integral'].fillna(0)
#61
final_combined_df['% de matrículas do ensino médio em tempo integral'] = final_combined_df['% de matrículas do ensino médio em tempo integral'].fillna(0)
#1191
final_combined_df['% de escolas da zona rural com oferta de EPT'] = final_combined_df['% de escolas da zona rural com oferta de EPT'].fillna(0)
#1191
final_combined_df['% de estudantes da rede estadual com frequência regular em 75%'] = final_combined_df['% de estudantes da rede estadual com frequência regular em 75%'].fillna(0)
#2275
final_combined_df['% de escolas da zona rural com oferta de Tempo Integral'] = final_combined_df['% de escolas da zona rural com oferta de Tempo Integral'].fillna(0)
"-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------"
#2230
final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'] = final_combined_df['Nº de escolas com oferta de tempo integral no ensino médio'].astype(int)
#2212
final_combined_df['Nº de matrículas de EJA integrado ao EPT'] = final_combined_df['Nº de matrículas de EJA integrado ao EPT'].astype(int)
#2213
final_combined_df['Nº de matrículas EPT'] = final_combined_df['Nº de matrículas EPT'].astype(int)
#67
final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'] = final_combined_df['Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais'].astype(int)
#2246
final_combined_df['Nº de matrículas do Ensino Médio'] = final_combined_df['Nº de matrículas do Ensino Médio'].astype(int)
#2208
final_combined_df['Nº de cursos EPT ofertados'] = final_combined_df['Nº de cursos EPT ofertados'].astype(int)
#2232
final_combined_df['Nº de matrículas da educação especial (AEE)'] = final_combined_df['Nº de matrículas da educação especial (AEE)'].astype(float).astype(int)
#72
final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'] = final_combined_df['Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC)'].astype(float).astype(int)
#2192
final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'] = final_combined_df['Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT)'].astype(float).astype(int)
#348
final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'] = final_combined_df['Nº de matrículas transferidas da rede estadual para a rede municipal'].astype(float).astype(int)
#65
final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'] = final_combined_df['Nº de escolas estaduais com oferta da Educação Profissional e Tecnológica (EPT)'].astype(float).astype(int)
#2196
final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados indígenas matriculados na rede'].astype(float).astype(int)
#2215
final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'] = final_combined_df['N° de estudantes autodeclarados pretos matriculados na rede'].astype(float).astype(int)
#2238
final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Matemática'].astype(float).astype(int)
#2237
final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'] = final_combined_df['Nota média dos estudantes da Rede no componente curricular Língua Portuguesa'].astype(float).astype(int)
#402
final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'] = final_combined_df['Média dos alunos da rede nas disciplinas de línguas estrangeiras'].astype(float).astype(int)
#382
final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'] = final_combined_df['Média nas avaliações das disciplinas especificas ITA/IME'].astype(float).astype(int)
#381
final_combined_df['Média nas avaliações nas demais disciplinas'] = final_combined_df['Média nas avaliações nas demais disciplinas'].astype(float).astype(int)

final_combined_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 32 columns):
 #   Column                                                                                      Non-Null Count  Dtype  
---  ------                                                                                      --------------  -----  
 0   Gerencia Regional                                                                           42 non-null     object 
 1   Ano                                                                                         42 non-null     int32  
 2   Nº total de matrículas da rede estadual de educação                                         42 non-null     int64  
 3   Nº de matrículas EPT                                                                        42 non-null     int32  
 4   Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais                 42 non-null     int32  
 5   Nº de matrículas do Ensino Médio             

66 - Indicador - % de matrículas de EPT sobre o total de matrículas - ok - A Validar

Mesclagem de DateFrame Parcial

In [160]:
# Realizar o merge entre os dois DataFrames com base nas colunas 'Id Escola' e 'Ano'
final_percent_df = pd.merge(combined_df_profissional, combined_df, on=['Gerencia Regional', 'Ano'])

# Criar uma nova coluna com a divisão das colunas 'Nº Matriculas Total (EPT)' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['Nº de matrículas EPT'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

final_percent_df['% de matrículas de EPT sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].apply(lambda x: f"{x:.4f}").astype(float)

final_percent_df = final_percent_df[["Gerencia Regional", "Ano", "% de matrículas de EPT sobre o total de matrículas"]]

# Mesclagem para DF final
final_percent_df = final_combined_df.merge(final_percent_df, on=['Gerencia Regional', 'Ano'], how='left')

# Exibir o DataFrame final combinado
print("Resultado final combinado:")
display(final_percent_df)

%run Caminhos.ipynb

# Salvar o DataFrame final em um arquivo CSV
#final_percent_df.to_csv(f'{pasta_hierarquia}{caminho_resultado_matriculas_gerencia}', index=False)
print("Arquivo CSV salvo com sucesso")

Resultado final combinado:


,Gerencia Regional,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,"% de matrículas de tempo integral no ensino regular, na rede estadual",Nº de matrículas de EJA integrado ao EPT,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de escolas da zona rural com oferta de Tempo Integral,% de matrículas de EPT sobre o total de matrículas
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2023,18344,6280,3590,0,43,73,1682,33,...,0.000000,1382,31,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3423
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2023,15283,4966,2459,0,47,68,1472,21,...,0.000000,930,18,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3249
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2023,6722,2104,1354,0,24,67,175,13,...,0.000000,409,10,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3130
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2023,12078,3873,2798,0,40,35,1138,15,...,0.000000,634,17,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3207
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2023,8210,2707,1606,0,25,22,739,16,...,0.000000,719,18,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3297
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2023,8197,3605,2584,0,34,43,508,18,...,0.000000,994,14,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4398
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2023,7110,2866,2034,0,32,54,696,14,...,0.000000,945,13,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4031
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2023,5281,1957,868,0,23,32,483,12,...,0.000000,282,15,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3706
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2023,15198,4203,2103,0,52,142,1257,24,...,0.000000,955,21,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.2765
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2023,4621,2193,1005,0,31,31,646,7,...,0.000000,479,10,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4746


Arquivo CSV salvo com sucesso


2266 - Indicador - % de matrículas de EJA integrado ao EPT (rede estadual)

In [161]:
# Criar uma nova coluna com a divisão das colunas 'Nº de matrículas de EJA integrado ao EPT' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas de EJA integrado ao EPT (rede estadual)'] = final_percent_df['Nº de matrículas de EJA integrado ao EPT'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

display(final_percent_df)

,Gerencia Regional,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,Nº de matrículas de EJA integrado ao EPT,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de escolas da zona rural com oferta de Tempo Integral,% de matrículas de EPT sobre o total de matrículas,% de matrículas de EJA integrado ao EPT (rede estadual)
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2023,18344,6280,3590,0,43,73,1682,33,...,1382,31,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3423,0.075338
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2023,15283,4966,2459,0,47,68,1472,21,...,930,18,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3249,0.060852
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2023,6722,2104,1354,0,24,67,175,13,...,409,10,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3130,0.060845
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2023,12078,3873,2798,0,40,35,1138,15,...,634,17,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3207,0.052492
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2023,8210,2707,1606,0,25,22,739,16,...,719,18,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3297,0.087576
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2023,8197,3605,2584,0,34,43,508,18,...,994,14,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4398,0.121264
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2023,7110,2866,2034,0,32,54,696,14,...,945,13,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4031,0.132911
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2023,5281,1957,868,0,23,32,483,12,...,282,15,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3706,0.053399
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2023,15198,4203,2103,0,52,142,1257,24,...,955,21,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.2765,0.062837
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2023,4621,2193,1005,0,31,31,646,7,...,479,10,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4746,0.103657


2207 - Indicadores - % de matrículas EPT no EM sobre o total de matrículas - ok - A Validar 

In [162]:
# Criar uma nova coluna com a divisão das colunas 'Nº de matrículas de EJA integrado ao EPT' e 'Nº total de matrículas da rede estadual de educação'
final_percent_df['% de matrículas EPT no EM sobre o total de matrículas'] = final_percent_df['Nº de matrículas do Ensino Médio'] / final_percent_df['Nº total de matrículas da rede estadual de educação']

final_percent_df['% de matrículas EPT no EM sobre o total de matrículas'] = final_percent_df['% de matrículas de EPT sobre o total de matrículas'].apply(lambda x: f"{x:.4f}").astype(float)

display(final_percent_df)

,Gerencia Regional,Ano,Nº total de matrículas da rede estadual de educação,Nº de matrículas EPT,Nº de matrículas na Educação de Jovens e Adultos (EJA) em escolas estaduais,Nº de matrículas do Ensino Médio,Nº de cursos EPT ofertados,Nº de matrículas da educação especial (AEE),Nº de matrículas nos cursos técnicos prioritários (SEDUCTEC),Nº de escolas com oferta da Educação de Jovens e Adultos (EJA) integrada com o (EPT),...,Nº de escolas com oferta de tempo integral no ensino médio,% de escolas de tempo integral com matrículas da Educação Profissional e Tecnológica (EPT),% de municípios com escolas da rede estadual em tempo integral,% de matrículas do ensino médio em tempo integral,% de estudantes da rede estadual com frequência regular em 75%,% de escolas da zona rural com oferta de EPT,% de escolas da zona rural com oferta de Tempo Integral,% de matrículas de EPT sobre o total de matrículas,% de matrículas de EJA integrado ao EPT (rede estadual),% de matrículas EPT no EM sobre o total de matrículas
0,GERENCIA REGIONAL DE EDUCACAO DA GRANDE TERESINA,2023,18344,6280,3590,0,43,73,1682,33,...,31,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3423,0.075338,0.3423
1,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2023,15283,4966,2459,0,47,68,1472,21,...,18,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3249,0.060852,0.3249
2,GERENCIA REGIONAL DE EDUCACAO DE BOM JESUS,2023,6722,2104,1354,0,24,67,175,13,...,10,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3130,0.060845,0.3130
3,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2023,12078,3873,2798,0,40,35,1138,15,...,17,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3207,0.052492,0.3207
4,GERENCIA REGIONAL DE EDUCACAO DE CORRENTE,2023,8210,2707,1606,0,25,22,739,16,...,18,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3297,0.087576,0.3297
5,GERENCIA REGIONAL DE EDUCACAO DE FLORIANO,2023,8197,3605,2584,0,34,43,508,18,...,14,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4398,0.121264,0.4398
6,GERENCIA REGIONAL DE EDUCACAO DE FRONTEIRAS,2023,7110,2866,2034,0,32,54,696,14,...,13,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4031,0.132911,0.4031
7,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2023,5281,1957,868,0,23,32,483,12,...,15,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.3706,0.053399,0.3706
8,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2023,15198,4203,2103,0,52,142,1257,24,...,21,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.2765,0.062837,0.2765
9,GERENCIA REGIONAL DE EDUCACAO DE PAULISTANA,2023,4621,2193,1005,0,31,31,646,7,...,10,1.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.4746,0.103657,0.4746


In [163]:
# Selecionar automaticamente todas as colunas para value_vars, exceto as que estão em id_vars
id_vars = ['Gerencia Regional', 'Ano']
value_vars = [col for col in final_percent_df.columns if col not in id_vars]

# Transformar o DataFrame para a forma PIVOT
long_df = final_percent_df.melt(id_vars=id_vars, 
                                 value_vars=value_vars,
                                 var_name='Indicador', value_name='Quantidade')

# Pivotar para obter as colunas separadas para os anos 2023 e 2024
pivot_df = long_df.pivot_table(index=['Gerencia Regional', 'Indicador', 'Ano'], values='Quantidade').reset_index()

# Renomear as colunas para torná-las mais legíveis
pivot_df.columns.name = None
pivot_df.rename(columns={2023: '2023', 2024: '2024'}, inplace=True)

# Seleciona as colunas "CasasDecimais" e "CodigoIndicador" do dataframe especificacao_superintendência_df
resumo_calculado = tabela_especificacao_ind_O_E[['Codigo Indicador', 'Indicador', 'CasasDecimais', 'UnidadeMedida', 'TipoIndicadorEstPro']]
resumo_calculado = resumo_calculado.drop_duplicates()

# Mesclar os DataFrames para adicionar as colunas com o cálculo
pivot_df = resumo_calculado.merge(pivot_df, on='Indicador', how='left')
# Salvando a tabela pivotada combinada em um arquivo CSV

pivot_df['CasasDecimais'] = pivot_df['CasasDecimais'].fillna(0)
pivot_df['CasasDecimais'] = pivot_df['CasasDecimais'].astype(int)
pivot_df['Codigo Indicador'] = pivot_df['Codigo Indicador'].fillna(0)
pivot_df['Codigo Indicador'] = pivot_df['Codigo Indicador'].astype(str)
pivot_df['Ano'] = pivot_df['Ano'].fillna(0)
pivot_df['Ano'] = pivot_df['Ano'].astype(int)
pivot_df = pivot_df[['Gerencia Regional', 'Codigo Indicador', 'Ano', 'Quantidade', 'TipoIndicadorEstPro']]

## IDENTIFICAR MOTIVO DE DUPLICAÇÃO DE LINHAS DO DATAFRAME
pivot_df = pivot_df.drop_duplicates(keep='first')

from datetime import datetime

# Capturar a data e hora atuais
data_hora_execucao = datetime.now().strftime('%d/%m/%Y %H:%M:%S')

# Extrair o ano atual
ano_atual = datetime.now().year

# Adicionar uma nova coluna ao DataFrame com a data e hora da última execução do script
pivot_df['Data Cadastro'] = data_hora_execucao# Condicional para definir 'Mes_Referente'
# Se o ano for igual ao ano atual, extrai o mês da data atual, senão atribui o valor 12
pivot_df['Mes_Referente'] = pivot_df.apply(
    lambda row: pd.to_datetime(data_hora_execucao, format='%d/%m/%Y %H:%M:%S').month 
                if row['Ano'] == ano_atual 
                else 12, axis=1
)

pivot_df['Mes_Referente'] = pivot_df['Mes_Referente'].astype(int)

pivot_df = pivot_df[['Gerencia Regional', 'Codigo Indicador', 'Ano', 'Quantidade', 'Data Cadastro', 'Mes_Referente', 'TipoIndicadorEstPro']]

# Exibindo a tabela final pivotada
#print("Tabela Final Pivotada:")
#display(pivot_df)
#pivot_df.info()

Preparação e definição de Base

In [8]:
# Preencher valores NaN na coluna 'Hierarquia' com string vazia
tabela_base_gerada['Hierarquia'] = tabela_base_gerada['Hierarquia'].fillna('')

# Filtro por escola
BASE_GERADA = tabela_base_gerada[(tabela_base_gerada['Hierarquia'].str.contains("Ger_Regional"))]

# Adicionar a nova coluna 'TipoIndicadorEstPro' com a informação 'Manual' em todas as linhas

BASE_GERADA = BASE_GERADA[['Gerencia Regional', 'Codigo Indicador', 'Ano', 'Quantidade', 'Data Cadastro', 'Mes_Referente', 'TipoIndicadorEstPro']]

BASE_GERADA['Gerencia Regional'] = BASE_GERADA['Gerencia Regional'].fillna(0)
BASE_GERADA['Gerencia Regional'] = BASE_GERADA['Gerencia Regional'].astype(str)

# Exibir o DataFrame
display(BASE_GERADA)
BASE_GERADA.info()


,Gerencia Regional,Codigo Indicador,Ano,Quantidade,Data Cadastro,Mes_Referente,TipoIndicadorEstPro
7376,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2236,2024,15.0,2024-11-01 11:00:21,12,None
7377,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2236,2024,12.0,2024-11-01 11:00:21,12,None
7378,GERENCIA REGIONAL DE EDUCACAO DE PIRIPIRI,2236,2024,4.0,2024-11-01 11:00:21,12,None
7379,GERENCIA REGIONAL DE EDUCACAO DE TERESINA,2236,2024,17.0,2024-11-01 11:00:21,12,None
7380,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2236,2024,4.0,2024-11-01 11:00:21,12,None
...,...,...,...,...,...,...,...
17536,GERENCIA REGIONAL DE EDUCACAO DE VALENCA,2281,2024,1.0,2024-05-12 14:39:07,11,E
17537,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2281,2024,1.0,2024-05-12 14:39:07,11,E
17538,GERENCIA REGIONAL DE EDUCACAO DE URUCUI,2281,2024,1.0,2024-05-12 14:39:07,11,E
17539,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 20ª,2281,2024,1.0,2024-05-12 14:39:07,11,E


<class 'pandas.core.frame.DataFrame'>
Index: 198 entries, 7376 to 17540
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   Gerencia Regional    198 non-null    object        
 1   Codigo Indicador     198 non-null    object        
 2   Ano                  198 non-null    int64         
 3   Quantidade           198 non-null    float64       
 4   Data Cadastro        198 non-null    datetime64[ns]
 5   Mes_Referente        198 non-null    int64         
 6   TipoIndicadorEstPro  166 non-null    object        
dtypes: datetime64[ns](1), float64(1), int64(2), object(3)
memory usage: 12.4+ KB


In [9]:
# Mesclar os DataFrames para adicionar as colunas com o cálculo
#base_indicadores = pd.concat([BASE_GERADA, pivot_df], ignore_index=True)
base_indicadores = BASE_GERADA
# Remover duplicatas baseadas em todas as colunas, mantendo apenas a última ocorrência
base_indicadores.drop_duplicates(keep='last', inplace=True)

display(base_indicadores)

,Gerencia Regional,Codigo Indicador,Ano,Quantidade,Data Cadastro,Mes_Referente,TipoIndicadorEstPro
7376,GERENCIA REGIONAL DE EDUCACAO DE PARNAIBA,2236,2024,15.0,2024-11-01 11:00:21,12,None
7377,GERENCIA REGIONAL DE EDUCACAO DE BARRAS,2236,2024,12.0,2024-11-01 11:00:21,12,None
7378,GERENCIA REGIONAL DE EDUCACAO DE PIRIPIRI,2236,2024,4.0,2024-11-01 11:00:21,12,None
7379,GERENCIA REGIONAL DE EDUCACAO DE TERESINA,2236,2024,17.0,2024-11-01 11:00:21,12,None
7380,GERENCIA REGIONAL DE EDUCACAO DE CAMPO MAIOR,2236,2024,4.0,2024-11-01 11:00:21,12,None
...,...,...,...,...,...,...,...
17536,GERENCIA REGIONAL DE EDUCACAO DE VALENCA,2281,2024,1.0,2024-05-12 14:39:07,11,E
17537,GERENCIA REGIONAL DE EDUCACAO DE OEIRAS,2281,2024,1.0,2024-05-12 14:39:07,11,E
17538,GERENCIA REGIONAL DE EDUCACAO DE URUCUI,2281,2024,1.0,2024-05-12 14:39:07,11,E
17539,GERENCIA REGIONAL DE EDUCACAO DE TERESINA - 20ª,2281,2024,1.0,2024-05-12 14:39:07,11,E


In [10]:
# O script atualiza diariamente um arquivo CSV acumulando dados e, no primeiro dia de cada mês, cria um backup mensal com os dados do último dia do mês anterior.

from datetime import datetime, timedelta
import os
import shutil

# Verificar se a pasta historico_diario existe, se não, criar
#historico_diario_path = 'historico_diario'
%run Caminhos.ipynb

# Caminhos para as pastas de histórico diário e mensal
historico_diario_path = os.path.join(pasta_hierarquia_historico, 'historico_diario')
historico_mensal_path = os.path.join(pasta_hierarquia_historico, 'historico_mensal')

# Verificar se a pasta historico_diario existe, se não, criar
if not os.path.exists(historico_diario_path):
    os.makedirs(historico_diario_path)

# Nome do arquivo CSV diário
arquivo_diario = os.path.join(historico_diario_path, 'dados_diario_ger_regional.csv')

# Verificar se o arquivo já existe
if os.path.exists(arquivo_diario):
    # Se o arquivo já existir, abrir em modo de apêndice e sem o cabeçalho
    base_indicadores.to_csv(arquivo_diario, mode='a', header=False, index=False)
else:
    # Se o arquivo não existir, criar um novo com o cabeçalho
    base_indicadores.to_csv(arquivo_diario, index=False)

# Verificar se hoje é o primeiro dia do mês
hoje = datetime.now()
if hoje.day == 1:
    # Obter a data do último dia do mês anterior
    ultimo_dia_mes_anterior = hoje.replace(day=1) - timedelta(days=1)
    
    # Verificar se o arquivo diário tem entradas para o último dia do mês anterior
    dados_diarios = pd.read_csv(arquivo_diario)
    dados_ultimo_dia = dados_diarios[dados_diarios['Data Cadastro'].str.startswith(ultimo_dia_mes_anterior.strftime('%Y-%m-%d'))]

    if not dados_ultimo_dia.empty:
        # Verificar se a pasta historico_mensal existe, se não, criar
        if not os.path.exists(historico_mensal_path):
            os.makedirs(historico_mensal_path)
        
        # Salvar os dados do último dia do mês anterior no arquivo mensal
        dados_ultimo_dia.to_csv(os.path.join(historico_mensal_path, f'dados_ger_regional{ultimo_dia_mes_anterior.strftime("%Y-%m-%d")}.csv'), index=False)